In [ ]:
import numpy as np
from astropy.table import Table
from astropy.table import join
from astropy.table import vstack
from astropy.table import Column

from astropy.io import fits
import pandas as pd

from astropy.coordinates import SkyCoord
import astropy.units as u

import matplotlib.pyplot as plt
import matplotlib
from matplotlib.colors import LogNorm
matplotlib.rcParams['figure.dpi'] = 360
matplotlib.rcParams['text.usetex'] = True
matplotlib.rcParams['text.latex.preamble'] = r'\usepackage{amsmath}'
# plt.style.use('dark_background')
import seaborn as sns

### LSS cat

In [ ]:
base = '/global/cfs/cdirs/desi/public/dr1/vac/dr1/lss/guadalupe/v1.0/LSScats/clustering'

In [ ]:
# !ls $base

In [ ]:
n = Table.read(f'{base}/BGS_BRIGHT_N_clustering.dat.fits')
s = Table.read(f'{base}/BGS_BRIGHT_S_clustering.dat.fits')
len(n), len(s)

In [ ]:
df = vstack([n,s]).to_pandas()
df.head()

In [ ]:
df.columns

In [ ]:
NBINS_RA = 220
NBINS_DEC = 110

RA_CENTER = 120.0

MAP3_MODE = 'mean_inv_weight_comp'
TITLE_PREFIX = 'DESI DR1 BGS'

USE_LOG_COUNTS = False
USE_LOG_WEIGHTED = True

In [ ]:
def wrap_angle_rad(x):
    return (x + np.pi) % (2.0 * np.pi) - np.pi


def ra_dec_to_mollweide(ra_deg, dec_deg, ra_center_deg=120.0):
    ra = np.asarray(ra_deg, dtype=float)
    dec = np.asarray(dec_deg, dtype=float)

    lon = np.deg2rad(-(ra - ra_center_deg))
    lon = wrap_angle_rad(lon)

    lat = np.deg2rad(dec)
    return lon, lat

In [ ]:
def make_edges():
    xedges = np.linspace(-np.pi, np.pi, NBINS_RA + 1)
    yedges = np.linspace(-np.pi/2, np.pi/2, NBINS_DEC + 1)
    return xedges, yedges

def compute_histogram(lon, lat, weights=None):
    xedges, yedges = make_edges()
    H, _, _ = np.histogram2d(lon, lat, bins=[xedges, yedges],
                             weights=weights)
    return H.T, xedges, yedges

In [ ]:
def compute_mean_map(lon, lat, values):
    xedges, yedges = make_edges()

    sum_map, _, _ = np.histogram2d(lon, lat,
                                   bins=[xedges, yedges],
                                   weights=values)

    count_map, _, _ = np.histogram2d(lon, lat,
                                     bins=[xedges, yedges])

    with np.errstate(divide='ignore', invalid='ignore'):
        mean_map = sum_map / count_map

    mean_map[count_map == 0] = np.nan
    return mean_map.T, xedges, yedges

In [ ]:
def make_ra_ticks(ra_center_deg=120.0):
    labels = np.array([270, 240, 210, 180, 150, 120, 90, 60, 30, 0, 330])
    positions_deg = -(labels - ra_center_deg)
    positions_rad = wrap_angle_rad(np.deg2rad(positions_deg))
    return positions_rad, [f'{x}°' for x in labels]


def setup_axis(ax, ra_center_deg=120.0):
    ax.grid(True, lw=0.4, alpha=0.5)

    xticks, xticklabels = make_ra_ticks(ra_center_deg=ra_center_deg)
    order = np.argsort(xticks)
    ax.set_xticks(xticks[order])
    ax.set_xticklabels(np.array(xticklabels)[order], fontsize=10)

    yticks_deg = np.arange(-75, 76, 15)
    ax.set_yticks(np.deg2rad(yticks_deg))
    ax.set_yticklabels([f'{y}°' for y in yticks_deg], fontsize=10)

    ax.set_xlabel(r'$\mathrm{RA} \, \mathrm{[deg]}$', fontsize=12, labelpad=12)
    ax.set_ylabel(r'$\mathrm{DEC} \, [\mathrm{deg}]$', fontsize=12)

In [ ]:
def plot_panel(ax, data, xedges, yedges, cbar_label, cmap,
               use_log=False, vmin=None, vmax=None, ra_center_deg=120.0):
    plot_data = data.copy()

    norm = None
    if use_log:
        positive = plot_data[np.isfinite(plot_data) & (plot_data > 0)]
        if len(positive) > 0:
            if vmin is None:
                vmin = positive.min()
            if vmax is None:
                vmax = positive.max()
            norm = LogNorm(vmin=vmin, vmax=vmax)

    mesh = ax.pcolormesh(xedges, yedges, plot_data,
                         shading='auto', cmap=cmap,
                         norm=norm,
                         vmin=None if norm is not None else vmin,
                         vmax=None if norm is not None else vmax,)

    setup_axis(ax, ra_center_deg=ra_center_deg)
    cbar = plt.colorbar(mesh, ax=ax, orientation='vertical', pad=0.05, shrink=0.82)
    cbar.set_label(cbar_label, fontsize=11)
    return mesh

In [ ]:
def galactic_plane_mollweide(ra_center_deg=120.0, npts=4000, jump_thresh=0.2):
    l = np.linspace(0.0, 360.0, npts)
    b = np.zeros_like(l)

    gal = SkyCoord(l=l * u.deg, b=b * u.deg, frame='galactic')
    eq = gal.icrs

    ra = eq.ra.deg
    dec = eq.dec.deg

    lon, lat = ra_dec_to_mollweide(ra, dec, ra_center_deg=ra_center_deg)

    dlon = np.abs(np.diff(lon))
    dlat = np.abs(np.diff(lat))

    bad = (dlon > jump_thresh) | (dlat > 0.2)

    lon_plot = lon.copy()
    lat_plot = lat.copy()

    lon_plot[1:][bad] = np.nan
    lat_plot[1:][bad] = np.nan

    return lon_plot, lat_plot

In [ ]:
required_cols = ['RA', 'DEC', 'WEIGHT', 'WEIGHT_COMP']

In [ ]:
mask = (np.isfinite(df['RA'].values) &
        np.isfinite(df['DEC'].values) &
        np.isfinite(df['WEIGHT'].values) &
        np.isfinite(df['WEIGHT_COMP'].values) &
        (df['WEIGHT_COMP'].values > 0))

In [ ]:
d = df.loc[mask].copy()

ra = d['RA'].values
dec = d['DEC'].values
weight = d['WEIGHT'].values
weight_comp = d['WEIGHT_COMP'].values

lon, lat = ra_dec_to_mollweide(ra, dec, ra_center_deg=RA_CENTER)
lon_gp, lat_gp = galactic_plane_mollweide(ra_center_deg=RA_CENTER)

In [ ]:
H_counts, xedges, yedges = compute_histogram(lon, lat, weights=None)

fig1 = plt.figure()
ax1 = fig1.add_subplot(projection='mollweide')

plot_panel(ax1, H_counts, xedges, yedges,
           cbar_label='Counts',
           cmap='Greens', use_log=USE_LOG_COUNTS,
           ra_center_deg=RA_CENTER)

ax1.plot(lon_gp, lat_gp, color='gray', lw=1.0, alpha=0.4)

plt.tight_layout()
plt.show()

In [ ]:
H_weight, xedges, yedges = compute_histogram(lon, lat, weights=weight)

fig2 = plt.figure()
ax2 = fig2.add_subplot(projection='mollweide')

plot_panel(ax2, H_weight, xedges, yedges,
           cbar_label='Weighted counts',
           cmap='Blues', use_log=USE_LOG_WEIGHTED,
           ra_center_deg=RA_CENTER)

ax2.plot(lon_gp, lat_gp, color='gray', lw=1.0, alpha=0.4)

plt.tight_layout()
plt.show()

In [ ]:
if MAP3_MODE == 'mean_weight_comp':
    values = weight_comp
    # title3 = f'{TITLE_PREFIX} — mean(WEIGHT_COMP) per bin'
    cbar3 = 'mean(WEIGHT_COMP)'
elif MAP3_MODE == 'mean_inv_weight_comp':
    values = 1.0 / weight_comp
    # title3 = f'{TITLE_PREFIX} — mean(1 / WEIGHT_COMP) per bin'
    cbar3 = 'mean(1 / WEIGHT_COMP)'

In [ ]:
H_mean, xedges, yedges = compute_mean_map(lon, lat, values)

fig3 = plt.figure()
ax3 = fig3.add_subplot(projection='mollweide')

plot_panel(ax3, H_mean, xedges, yedges,
           cbar_label=cbar3,
           cmap='Oranges', use_log=False,
           ra_center_deg=RA_CENTER)

ax3.plot(lon_gp, lat_gp, color='gray', lw=1.0, alpha=0.4)

plt.tight_layout()
plt.show()

### dr1/survey/ops/surveyops/tags/1.0/ops/

In [ ]:
base = '/global/cfs/cdirs/desi/public/dr1/survey/ops/surveyops/tags/1.0/ops'

In [ ]:
tab = Table.read(f'{base}/tiles-specstatus.ecsv', format='ascii.ecsv')
df = tab.to_pandas()
df.head()

In [ ]:
df.columns

In [ ]:
for col in ['SURVEY', 'FAPRGRM', 'FAFLAVOR', 'GOALTYPE', 'OBSSTATUS', 'ZDONE', 'QA']:
    print('\n', col)
    print(df[col].value_counts(dropna=False).head())

In [ ]:
print(df[['EFFTIME_SPEC', 'GOALTIME', 'BGS_EFFTIME_BRIGHT',
          'ELG_EFFTIME_DARK', 'LRG_EFFTIME_DARK', 'LYA_EFFTIME_DARK']].describe())

In [ ]:
cols = ['TILEID', 'FAPRGRM', 'GOALTYPE', 'OBSSTATUS', 'ZDONE',
        'EFFTIME_SPEC', 'GOALTIME', 'BGS_EFFTIME_BRIGHT']
print(df[cols].head().to_string())

---

In [ ]:
dfm = df[df['SURVEY'] == 'main']

In [ ]:
dfm['COMP_TILE'] = np.nan

m_bright = dfm['GOALTYPE'] == 'bright'
m_dark   = dfm['GOALTYPE'] == 'dark'
m_backup = dfm['GOALTYPE'] == 'backup'

In [ ]:
dfm.loc[m_bright, 'COMP_TILE'] = (dfm.loc[m_bright, 'BGS_EFFTIME_BRIGHT'] /
                                  dfm.loc[m_bright, 'GOALTIME'])

dfm.loc[m_dark, 'COMP_TILE'] = (dfm.loc[m_dark, 'EFFTIME_SPEC'] /
                                dfm.loc[m_dark, 'GOALTIME'])

dfm.loc[m_backup, 'COMP_TILE'] = (dfm.loc[m_backup, 'EFFTIME_SPEC'] /
                                  dfm.loc[m_backup, 'GOALTIME'])

dfm['COMP_TILE'] = dfm['COMP_TILE'].clip(lower=0, upper=1)

In [ ]:
df_bright = dfm[dfm['GOALTYPE'] == 'bright']
df_dark = dfm[dfm['GOALTYPE'] == 'dark']
df_backup = dfm[dfm['GOALTYPE'] == 'backup']

In [ ]:
def wrap_angle_rad(x):
    return (x + np.pi) % (2.0 * np.pi) - np.pi

def ra_dec_to_mollweide(ra_deg, dec_deg, ra_center_deg=120.0):
    ra = np.asarray(ra_deg, dtype=float)
    dec = np.asarray(dec_deg, dtype=float)

    lon = np.deg2rad(-(ra - ra_center_deg))
    lon = wrap_angle_rad(lon)
    lat = np.deg2rad(dec)

    return lon, lat

In [ ]:
def make_ra_ticks(ra_center_deg=120.0):
    labels = np.array([270, 240, 210, 180, 150, 120, 90, 60, 30, 0, 330])
    positions_deg = -(labels - ra_center_deg)
    positions_rad = wrap_angle_rad(np.deg2rad(positions_deg))
    return positions_rad, [f'{x}°' for x in labels]

def setup_axis(ax, title, ra_center_deg=120.0):
    ax.set_title(title, fontsize=15, pad=12)
    ax.grid(lw=0.4, alpha=0.5)

    xticks, xticklabels = make_ra_ticks(ra_center_deg=ra_center_deg)
    order = np.argsort(xticks)
    ax.set_xticks(xticks[order])
    ax.set_xticklabels(np.array(xticklabels)[order], fontsize=10)

    yticks_deg = np.arange(-75, 76, 15)
    ax.set_yticks(np.deg2rad(yticks_deg))
    ax.set_yticklabels([f'{y}°' for y in yticks_deg], fontsize=10)

    ax.set_xlabel(r'$\mathrm{RA} \, \mathrm{[deg]}$', fontsize=12, labelpad=12)
    ax.set_ylabel(r'$\mathrm{DEC} \, [\mathrm{deg}]$', fontsize=12)

In [ ]:
RA_CENTER = 120.0

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(13, 14),
                         subplot_kw={'projection': 'mollweide'})

datasets = [('Bright Completeness', df_bright, 'Greens'),
            ('Dark Completeness',   df_dark,   'Blues'),
            ('Backup Completeness', df_backup, 'Oranges'),]

for ax, (title, dfi, cmap) in zip(axes, datasets):
    lon, lat = ra_dec_to_mollweide(dfi['TILERA'].values,
                                   dfi['TILEDEC'].values,
                                   ra_center_deg=RA_CENTER)

    sc = ax.scatter(lon, lat,
                    c=dfi['COMP_TILE'].values,
                    s=20, cmap=cmap, vmin=0,
                    vmax=1, edgecolors='none',
                    alpha=0.7)

    setup_axis(ax, title, ra_center_deg=RA_CENTER)

    cbar = plt.colorbar(sc, ax=ax, orientation='vertical', pad=0.04, shrink=0.85)
    cbar.set_label('Completeness', fontsize=11)

plt.tight_layout()
plt.show()

In [ ]:
def compute_mean_binned_map(lon, lat, values, nbins_ra=180, nbins_dec=90):
    xedges = np.linspace(-np.pi, np.pi, nbins_ra + 1)
    yedges = np.linspace(-np.pi/2, np.pi/2, nbins_dec + 1)

    sum_map, _, _ = np.histogram2d(lon, lat,
                                   bins=[xedges, yedges],
                                   weights=values)

    count_map, _, _ = np.histogram2d(lon, lat,
                                     bins=[xedges, yedges])

    with np.errstate(divide='ignore', invalid='ignore'):
        mean_map = sum_map / count_map

    mean_map[count_map == 0] = np.nan

    return mean_map.T, xedges, yedges

In [ ]:
RA_CENTER = 120.0

NBINS_RA = 240
NBINS_DEC = 120

In [ ]:
datasets = [('Bright Completeness', df_bright, 'Greens'),
            ('Dark Completeness',   df_dark,   'Blues'),
            ('Backup Completeness', df_backup, 'Oranges'),]

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(13, 14),
                         subplot_kw={'projection': 'mollweide'})

for ax, (title, dfi, cmap) in zip(axes, datasets):
    lon, lat = ra_dec_to_mollweide(dfi['TILERA'].values,
                                   dfi['TILEDEC'].values,
                                   ra_center_deg=RA_CENTER)

    mean_map, xedges, yedges = compute_mean_binned_map(lon, lat,
                                                       dfi['COMP_TILE'].values,
                                                       nbins_ra=NBINS_RA,
                                                       nbins_dec=NBINS_DEC)

    mesh = ax.pcolormesh(xedges, yedges, mean_map,
                         shading='auto', cmap=cmap,
                         vmin=0, vmax=1)

    setup_axis(ax, title, ra_center_deg=RA_CENTER)

    cbar = plt.colorbar(mesh, ax=ax, orientation='vertical', pad=0.04, shrink=0.85)
    cbar.set_label('Completeness', fontsize=11)

plt.tight_layout()
plt.show()

----

In [ ]:
N_RA = 720
N_DEC = 240

TILE_RADIUS_DEG = 1.6

USE_DR1_CUTOFF = True
DR1_LASTNIGHT = 20220613

In [ ]:
def angular_separation_deg(ra1_deg, dec1_deg, ra2_deg, dec2_deg):
    ra1 = np.deg2rad(ra1_deg)
    dec1 = np.deg2rad(dec1_deg)
    ra2 = np.deg2rad(ra2_deg)
    dec2 = np.deg2rad(dec2_deg)

    cosang = (np.sin(dec1) * np.sin(dec2) +
              np.cos(dec1) * np.cos(dec2) * np.cos(ra1 - ra2))
    cosang = np.clip(cosang, -1.0, 1.0)
    return np.rad2deg(np.arccos(cosang))

In [ ]:
ra_centers = np.linspace(0.0, 360.0, N_RA, endpoint=False)
dec_centers = np.linspace(-90.0, 90.0, N_DEC)

RA_grid, DEC_grid = np.meshgrid(ra_centers, dec_centers)

In [ ]:
ra_edges = np.linspace(0.0, 360.0, N_RA + 1)
dec_edges = np.linspace(-90.0, 90.0, N_DEC + 1)

In [ ]:
RA_edges_2d, DEC_edges_2d = np.meshgrid(ra_edges[:-1], dec_edges[:-1])
lon_edges, lat_edges = ra_dec_to_mollweide(ra_edges, dec_edges, ra_center_deg=RA_CENTER)

In [ ]:
LON_edges_2d, LAT_edges_2d = np.meshgrid(lon_edges, lat_edges)

In [ ]:
def angular_separation_deg(ra1_deg, dec1_deg, ra2_deg, dec2_deg):
    ra1 = np.deg2rad(ra1_deg)
    dec1 = np.deg2rad(dec1_deg)
    ra2 = np.deg2rad(ra2_deg)
    dec2 = np.deg2rad(dec2_deg)

    cosang = (np.sin(dec1) * np.sin(dec2) +
              np.cos(dec1) * np.cos(dec2) * np.cos(ra1 - ra2))
    cosang = np.clip(cosang, -1.0, 1.0)
    return np.rad2deg(np.arccos(cosang))

In [ ]:
def build_tile_disk_map(dfi, ra_grid, dec_grid, tile_radius_deg=1.6):
    sum_map = np.zeros_like(ra_grid, dtype=np.float64)
    hit_map = np.zeros_like(ra_grid, dtype=np.int32)

    ra_all = ra_grid
    dec_all = dec_grid

    for _, row in dfi.iterrows():
        ra0 = float(row['TILERA'])
        dec0 = float(row['TILEDEC'])
        comp = float(row['COMP_TILE'])

        dec_min = dec0 - tile_radius_deg
        dec_max = dec0 + tile_radius_deg

        dec_mask = (dec_all[:, 0] >= dec_min) & (dec_all[:, 0] <= dec_max)
        if not np.any(dec_mask):
            continue

        cosdec = np.cos(np.deg2rad(dec0))
        if cosdec < 1e-3:
            dra = 180.0
        else:
            dra = tile_radius_deg / cosdec
            dra = min(dra, 180.0)

        ra_low = ra0 - dra
        ra_high = ra0 + dra

        ra_vals = ra_all[0, :]

        if ra_low < 0:
            ra_mask = (ra_vals >= (ra_low % 360.0)) | (ra_vals <= ra_high)
        elif ra_high >= 360:
            ra_mask = (ra_vals >= ra_low) | (ra_vals <= (ra_high % 360.0))
        else:
            ra_mask = (ra_vals >= ra_low) & (ra_vals <= ra_high)

        if not np.any(ra_mask):
            continue

        sub_ra = ra_all[np.ix_(dec_mask, ra_mask)]
        sub_dec = dec_all[np.ix_(dec_mask, ra_mask)]

        sep = angular_separation_deg(sub_ra, sub_dec, ra0, dec0)
        covered = sep <= tile_radius_deg

        if np.any(covered):
            sum_map[np.ix_(dec_mask, ra_mask)] += covered * comp
            hit_map[np.ix_(dec_mask, ra_mask)] += covered.astype(np.int32)

    with np.errstate(divide='ignore', invalid='ignore'):
        mean_map = sum_map / hit_map
    mean_map[hit_map == 0] = np.nan

    return mean_map, hit_map

In [ ]:
bright_map, bright_hits = build_tile_disk_map(df_bright, RA_grid, DEC_grid, tile_radius_deg=TILE_RADIUS_DEG)
dark_map, dark_hits = build_tile_disk_map(df_dark, RA_grid, DEC_grid, tile_radius_deg=TILE_RADIUS_DEG)
backup_map, backup_hits = build_tile_disk_map(df_backup, RA_grid, DEC_grid, tile_radius_deg=TILE_RADIUS_DEG)

In [ ]:
LON_plot, LAT_plot = np.meshgrid(lon_edges, lat_edges)

In [ ]:
panels = [('Bright Completeness', bright_map, 'Greens'),
          ('Dark Completeness',   dark_map,   'Blues'),
          # ('Backup Completeness', backup_map, 'Oranges')
         ]

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 15),
                         subplot_kw={'projection': 'mollweide'})

for ax, (title, map2d, cmap) in zip(axes, panels):
    mesh = ax.pcolormesh(LON_plot, LAT_plot, map2d,
                         shading='auto', cmap=cmap,
                         vmin=0.0, vmax=1.0, alpha=0.7)

    setup_axis(ax, title, ra_center_deg=RA_CENTER)
    ax.plot(lon_gp, lat_gp, color='gray', lw=1.0, alpha=0.4)

    cbar = plt.colorbar(mesh, ax=ax, orientation='vertical', pad=0.04, shrink=0.85)
    cbar.set_label('Completeness', fontsize=11)

plt.tight_layout()
plt.show()

---

## Mask

In [ ]:
df_bright

In [ ]:
df_bright.shape

In [ ]:
df_bright[df_bright['COMP_TILE']>0.8].shape

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(13, 14),
                         subplot_kw={'projection': 'mollweide'})

datasets = [('Bright Completeness', df_bright, 'Greens'),
            ('Dark Completeness',   df_dark,   'Blues'),
            ('Backup Completeness', df_backup, 'Oranges'),]

for ax, (title, dfi, cmap) in zip(axes, datasets):
    lon, lat = ra_dec_to_mollweide(dfi['TILERA'].values,
                                   dfi['TILEDEC'].values,
                                   ra_center_deg=RA_CENTER)
    
    sc = ax.scatter(lon, lat,
                    c=dfi['COMP_TILE'].values,
                    s=20, cmap=cmap, vmin=0,
                    vmax=1, edgecolors='none',
                    alpha=0.7)

    df2 = dfi[dfi['COMP_TILE']<0.7]
    lon, lat = ra_dec_to_mollweide(df2['TILERA'].values,
                                   df2['TILEDEC'].values,
                                   ra_center_deg=RA_CENTER)
    ax.scatter(lon, lat,
                    s=20, color='red')

    setup_axis(ax, title, ra_center_deg=RA_CENTER)

    cbar = plt.colorbar(sc, ax=ax, orientation='vertical', pad=0.04, shrink=0.85)
    cbar.set_label('Completeness', fontsize=11)

plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(14, 5),
                         subplot_kw={'projection': 'mollweide'})

# datasets = [('Bright Completeness', df_bright, 'Greens')]
title, dfi, cmap = 'Bright Completeness $>0.9$', df_bright, 'Greens'

df2 = dfi[dfi['COMP_TILE']>0.9]
lon, lat = ra_dec_to_mollweide(df2['TILERA'].values,
                               df2['TILEDEC'].values,
                               ra_center_deg=RA_CENTER)
sc = ax.scatter(lon, lat,
                c=df2['COMP_TILE'].values,
                s=20, cmap=cmap, vmin=0,
                vmax=1, edgecolors='none',
                alpha=0.7)

setup_axis(ax, title, ra_center_deg=RA_CENTER)

cbar = plt.colorbar(sc, ax=ax, orientation='vertical', pad=0.04, shrink=0.85)
cbar.set_label('Completeness', fontsize=11)

plt.tight_layout()
plt.show()

--------------

In [ ]:
base = '/global/cfs/cdirs/desi/public/dr1/vac/dr1/lss/guadalupe/v1.0/LSScats/clustering'

In [ ]:
n = Table.read(f'{base}/BGS_BRIGHT_N_clustering.dat.fits')
s = Table.read(f'{base}/BGS_BRIGHT_S_clustering.dat.fits')
df = vstack([n,s]).to_pandas()

In [ ]:
df.columns

In [ ]:
df['NTILE'].describe()

In [ ]:
df['WEIGHT'].describe()

In [ ]:
fig, ax = plt.subplots()
ax.grid(lw=0.25)

ax.hist(df['WEIGHT'], color='orange', edgecolor='darkorange', linewidth=1.)
ax.set_yscale('log')
ax.set_ylabel('Counts')
ax.set_xlabel('WEIGHT')

plt.show()

In [ ]:
fig, ax = plt.subplots()
ax.grid(lw=0.25)

ax.hist(df['WEIGHT_COMP'], color='royalblue', edgecolor='navy', linewidth=1.)
ax.set_yscale('log')
ax.set_ylabel('Counts')
ax.set_xlabel('WEIGHT_COMP')

plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(14, 5),
                         subplot_kw={'projection': 'mollweide'})

title = 'Bright Completeness'
lon, lat = ra_dec_to_mollweide(df['RA'].values,
                               df['DEC'].values,
                               ra_center_deg=RA_CENTER)
sc = ax.scatter(lon, lat,
                c=df['WEIGHT_COMP'].values,
                s=1, cmap='Greens_r',
                vmin=0.7, vmax=56,
                edgecolors='none')

setup_axis(ax, title, ra_center_deg=RA_CENTER)

cbar = plt.colorbar(sc, ax=ax, orientation='vertical', pad=0.04, shrink=0.85)
cbar.set_label('WEIGHT_COMP', fontsize=11)

plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(14, 5),
                         subplot_kw={'projection': 'mollweide'})

title = 'Bright Completeness'
lon, lat = ra_dec_to_mollweide(df['RA'].values,
                               df['DEC'].values,
                               ra_center_deg=RA_CENTER)
sc = ax.scatter(lon, lat,
                c=df['WEIGHT_COMP'].values,
                s=1, cmap='Greens_r',
                # vmin=0, vmax=1,
                edgecolors='none',
                norm=LogNorm(vmin=df['WEIGHT_COMP'].min(), vmax=df['WEIGHT_COMP'].max()),)
                # alpha=0.7)

setup_axis(ax, title, ra_center_deg=RA_CENTER)

cbar = plt.colorbar(sc, ax=ax, orientation='vertical', pad=0.04, shrink=0.85)
cbar.set_label('WEIGHT_COMP', fontsize=11)

plt.tight_layout()
plt.show()

In [ ]:
dff = df[df['WEIGHT_COMP']<5]
df.shape, dff.shape

In [ ]:
dff['WEIGHT_COMP'].describe()

In [ ]:
fig, ax = plt.subplots(figsize=(14, 5),
                         subplot_kw={'projection': 'mollweide'})

title = 'Bright Completeness'
lon, lat = ra_dec_to_mollweide(dff['RA'].values,
                               dff['DEC'].values,
                               ra_center_deg=RA_CENTER)
sc = ax.scatter(lon, lat,
                # c=dff['WEIGHT_COMP'].values,
                s=1, c='royalblue')
                # cmap='Greens_r',
                # vmin=1., vmax=9.,
                # edgecolors='none')

setup_axis(ax, title, ra_center_deg=RA_CENTER)

# cbar = plt.colorbar(sc, ax=ax, orientation='vertical', pad=0.04, shrink=0.85)
# cbar.set_label('WEIGHT_COMP', fontsize=11)

plt.tight_layout()
plt.show()